# 📈 Notebook 02: Kiểm tra và Đánh giá Mô hình Baseline (TF-IDF)
**Đề tài:** Phân tích cảm xúc theo khía cạnh trên UIT-ViSFD.

Mục tiêu của notebook:
1. Đọc kết quả đánh giá của mô hình Baseline trên tập Dev (`outputs/metrics/baseline_dev.json`).
2. Trực quan hóa **Aspect F1** và **Joint F1** trên từng khía cạnh.
3. Khảo sát ma trận nhầm lẫn (Confusion Matrix) và độ chính xác của dự đoán phân loại cực tính.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")

from src.data.schema import ASPECTS, POLARITES
from src.data.load import load_split

sns.set_theme(style="whitegrid")

## 1. Nạp metrics của Baseline từ outputs/metrics/baseline_dev.json

In [ ]:
metrics_path = Path("../outputs/metrics/baseline_dev.json")
with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("=== TỔNG QUAN CHỈ SỐ BASELINE TRÊN DEV ===")
print(f"• Aspect Micro-F1 : {metrics.get('aspect_micro_f1', 0):.4f}")
print(f"• Aspect Macro-F1 : {metrics.get('aspect_macro_f1', 0):.4f}")
print(f"• Sentiment Macro : {metrics.get('sentiment_macro_f1_given_gold_aspect', 0):.4f}")
print(f"• JOINT F1        : {metrics.get('joint_f1', 0):.4f}")
print(f"• Ngưỡng tối ưu   : {metrics.get('threshold', 0):.2f}")

## 2. So sánh Aspect F1 vs Joint F1 trên từng khía cạnh

In [ ]:
asp_f1 = metrics.get("aspect_f1_per_aspect", {})
joint_f1 = metrics.get("joint_f1_per_aspect", {})

df_comp = pd.DataFrame({
    "Khía cạnh": ASPECTS,
    "Aspect F1 (Phát hiện)": [asp_f1.get(a, 0) for a in ASPECTS],
    "Joint F1 (End-to-End)": [joint_f1.get(a, 0) for a in ASPECTS]
})

df_comp_melted = df_comp.melt(id_vars=["Khía cạnh"], var_name="Loại Metric", value_name="F1-Score")

plt.figure(figsize=(12, 5))
sns.barplot(data=df_comp_melted, x="Khía cạnh", y="F1-Score", hue="Loại Metric", palette=["#3498db", "#e67e22"])
plt.title("So sánh Aspect F1 và Joint F1 trên từng khía cạnh của Baseline")
plt.ylim(0, 1.0)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Xem trước một số dự đoán trên tập Dev từ baseline_dev.jsonl

In [ ]:
preds_path = Path("../outputs/predictions/baseline_dev.jsonl")
with open(preds_path, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f][:5]

for s in samples:
    print("-" * 60)
    print(f"ID  : {s['id']}")
    print(f"Câu : {s['text']}")
    print(f"Pred: {s['pred']}")